# California Housing Dataset

Implementación con PyTorch.

In [169]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import torch.optim as optim


from torch import nn
from torch.utils.data import random_split, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset

In [170]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

# Obteniendo el dataset y partición

In [171]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

# Clases y funciones usadas

In [172]:
def compute_preprocessing_params(X_train, feature_mask):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    mask = feature_mask
    X_selected = X_train_tensor[:, mask]

    # Scaling
    means = X_selected.mean(dim=0)
    stds = X_selected.std(dim=0)

    # Trimming
    Q1 = torch.quantile(X_selected, 0.25, dim=0)
    Q3 = torch.quantile(X_selected, 0.75, dim=0)
    IQR = Q3 - Q1

    lower_bounds = Q1 - 1.5 * IQR
    upper_bounds = Q3 + 1.5 * IQR

    return lower_bounds, upper_bounds, means, stds

In [173]:
class ScalingLayer(nn.Module):

    def __init__(self, mean: torch.Tensor, std: torch.Tensor, feature_mask: torch.Tensor):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        self.register_buffer('feature_mask', feature_mask)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Para no modificar el original
        X_scaled = X.clone()

        mask = self.feature_mask

        X_scaled[:, mask] = (X_scaled[:, mask] - self.mean) / (self.std + 1e-8)

        return X_scaled

In [174]:
class OutlierTrimmingLayer(nn.Module):

    def __init__(self, lower_bounds: torch.Tensor, upper_bounds: torch.Tensor, feature_mask: torch.Tensor):
        super().__init__()
        self.register_buffer('lower_bounds', lower_bounds)
        self.register_buffer('upper_bounds', upper_bounds)
        self.register_buffer('feature_mask', feature_mask)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        # Para no modificar el original
        X_trimmed = X.clone()

        mask = self.feature_mask

        X_trimmed[:, mask] = torch.clamp(
            X_trimmed[:, mask],
            self.lower_bounds,
            self.upper_bounds
        )

        return X_trimmed

In [175]:
class CaliforniaHousingNN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size, lower_bound, upper_bound, mean, std, feature_mask):
        super().__init__()

        # Preprocessing
        self.trim_layer = OutlierTrimmingLayer(
            lower_bound,
            upper_bound,
            feature_mask
        )
        self.scale_layer = ScalingLayer(
            mean,
            std,
            feature_mask
        )

        self.hidden = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        # Preprocessing
        x = self.trim_layer(x)
        x = self.scale_layer(x)

        # Main stuff
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [176]:
class CaliforniaHousingNN_ALT(nn.Module):

    def __init__(self, input_size, hidden_size, hidden_size_dos, output_size, lower_bound, upper_bound, mean, std, feature_mask):
        super().__init__()

        self.trim_layer = OutlierTrimmingLayer(
            lower_bound,
            upper_bound,
            feature_mask
        )
        self.scale_layer = ScalingLayer(
            mean,
            std,
            feature_mask
        )

        self.hidden = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.hidden2 = nn.Linear(hidden_size, hidden_size_dos)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):

        x = self.trim_layer(x)
        x = self.scale_layer(x)

        x = self.hidden(x)
        x = self.relu(x)
        x = self.hidden2(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [177]:
def train(model, num_epochs, optimizer, batch_size):

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    loss_fn = nn.MSELoss()

    # Train
    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []

        for X_i, y_i in val_loader:
                y_pred = model(X_i)
                val_loss += loss_fn(y_pred, y_i).item()
                all_preds.append(y_pred)
                all_targets.append(y_i)

        y_pred_all = torch.cat(all_preds).flatten()
        y_true_all = torch.cat(all_targets).flatten()

        # R^2
        ss_res = torch.sum((y_true_all - y_pred_all) ** 2)
        ss_tot = torch.sum((y_true_all - torch.mean(y_true_all)) ** 2)
        r2 = 1 - ss_res / (ss_tot + 1e-8)

        # RMSE
        rmse = torch.sqrt(torch.mean((y_true_all - y_pred_all) ** 2))

        print(f"Epoch: {epoch+1} \tLoss: {val_loss/len(val_loader):.4f} \tR²: {r2:.4f} \tRMSE: ${rmse.item() * 100000:.2f}")

# Proceso principal (carga de datos)

In [178]:
# Convertir a tensores
X_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
X_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
X_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

# Masking para ignorar Latitude y Longitude
features_to_process = [0, 1, 2, 3, 4, 5]
feature_mask = torch.tensor([True if i in features_to_process else False for i in range(8)])

lower_bounds, upper_bounds, means, stds = compute_preprocessing_params(x_train.values, feature_mask)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)


# Modelo 1

In [179]:
# Train model 1
model_m1 = CaliforniaHousingNN(8, 100, 1, lower_bounds, upper_bounds, means, stds, feature_mask)
optimizer_m1 = optim.SGD(model_m1.parameters(), lr=0.01)
batch_size_m1 = 64
epochs_m1 = 12
train(model_m1, epochs_m1, optimizer_m1, batch_size_m1)

Epoch: 1 	Loss: 1.7560 	R²: -0.3255 	RMSE: $131436.69
Epoch: 2 	Loss: 1.3166 	R²: -0.0003 	RMSE: $114181.90
Epoch: 3 	Loss: 1.3162 	R²: -0.0001 	RMSE: $114170.42
Epoch: 4 	Loss: 1.3158 	R²: -0.0000 	RMSE: $114165.25
Epoch: 5 	Loss: 1.3157 	R²: -0.0000 	RMSE: $114166.57
Epoch: 6 	Loss: 1.3159 	R²: -0.0000 	RMSE: $114165.75
Epoch: 7 	Loss: 1.3159 	R²: -0.0000 	RMSE: $114166.07
Epoch: 8 	Loss: 1.3165 	R²: -0.0002 	RMSE: $114177.10
Epoch: 9 	Loss: 1.3161 	R²: -0.0001 	RMSE: $114168.52
Epoch: 10 	Loss: 1.3160 	R²: -0.0000 	RMSE: $114167.56
Epoch: 11 	Loss: 1.3158 	R²: -0.0006 	RMSE: $114196.54
Epoch: 12 	Loss: 1.3156 	R²: -0.0001 	RMSE: $114170.05
